# Dashboard Design Recommendation – Fine-Tuning Notebook
**Master Thesis | PEFT/QLoRA Fine-Tuning with Qwen2.5-0.5B-Instruct**

## What this notebook does:
1. Checks GPU and installs all required libraries
2. Writes project files (config, utils, scripts) directly into Colab
3. Generates a synthetic training dataset (80 train / 10 val / 10 test)
4. Fine-tunes Qwen2.5-0.5B-Instruct with QLoRA (LoRA on 4-bit quantized model)
5. Runs inference and evaluates the fine-tuned model

## Requirements:
- **Runtime**: GPU (T4 or better) → Runtime > Change runtime type > GPU
- **VRAM**: ~12 GB (T4 has 15 GB – sufficient)
- **Time**: ~15–30 minutes for full training

> 💡 **Tip**: File > Save a copy in Drive to keep your work!

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('❌ No GPU found!')
    print('Go to: Runtime > Change runtime type > GPU')


## Step 1: Install Libraries

This takes 3–5 minutes on first run.

In [ ]:
!pip install -q transformers==4.40.0 datasets peft trl bitsandbytes accelerate
!pip install -q pyyaml jsonlines rouge-score evaluate
print('✅ Installation complete!')


## Step 2: Write Project Files

We write all necessary files directly into the Colab environment.

In [ ]:
config_yaml = '''
model:
  name: "Qwen/Qwen2.5-0.5B-Instruct"
  max_seq_length: 2048
  load_in_4bit: true
  load_in_8bit: false
lora:
  r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  bias: "none"
  task_type: "CAUSAL_LM"
  target_modules: ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
training:
  output_dir: "./outputs/checkpoints"
  num_train_epochs: 3
  per_device_train_batch_size: 2
  gradient_accumulation_steps: 4
  learning_rate: 2.0e-4
  lr_scheduler_type: "cosine"
  warmup_ratio: 0.1
  weight_decay: 0.01
  fp16: true
  bf16: false
  logging_steps: 10
  save_steps: 50
  save_total_limit: 2
  evaluation_strategy: "no"
  report_to: "none"
  seed: 42
data:
  train_file: "./data/train.jsonl"
  val_file: "./data/val.jsonl"
  test_file: "./data/test.jsonl"
  max_samples: null
dataset_generation:
  num_train_samples: 80
  num_val_samples: 10
  num_test_samples: 10
  seed: 42
inference:
  adapter_path: "./outputs/final_model"
  max_new_tokens: 1024
  temperature: 0.1
  top_p: 0.9
  do_sample: true
paths:
  outputs_dir: "./outputs"
  checkpoints_dir: "./outputs/checkpoints"
  final_model_dir: "./outputs/final_model"
  logs_dir: "./outputs/logs"
  results_dir: "./outputs/results"
'''
with open('config.yaml', 'w') as f:
    f.write(config_yaml)
print('✅ config.yaml written')


In [ ]:
import os
os.makedirs('utils', exist_ok=True)
with open('utils/__init__.py', 'w') as f:
    f.write('# utils package\n')
print('✅ utils/ directory created')
print('⚠️  Now upload utils/helpers.py from your local project,')
print('   or paste its content into a new cell below.')


In [ ]:
# Upload utils/helpers.py and other scripts from your local machine
from google.colab import files
print('Select files to upload: utils/helpers.py, generate_dataset.py, train.py, inference.py')
uploaded = files.upload()
# Move helpers.py into utils/
import shutil
if 'helpers.py' in uploaded:
    shutil.move('helpers.py', 'utils/helpers.py')
    print('✅ utils/helpers.py moved correctly')
print('Uploaded files:', list(uploaded.keys()))


## Step 3: Generate Synthetic Dataset

In [ ]:
!python generate_dataset.py --config config.yaml

# Verify
import os
for fname in ['data/train.jsonl', 'data/val.jsonl', 'data/test.jsonl']:
    if os.path.exists(fname):
        with open(fname) as fp:
            n = sum(1 for _ in fp)
        print(f'  ✅ {fname}: {n} examples')
    else:
        print(f'  ❌ MISSING: {fname}')


In [ ]:
import json
with open('data/train.jsonl') as f:
    sample = json.loads(f.readline())
print('=== BRIEF ===')
print(json.dumps(sample['brief'], indent=2))
print('\n=== CONTEXT SUMMARY ===')
print(sample['recommendation']['context_summary'])
print('\n=== KPI → CHART MAPPING ===')
for m in sample['recommendation']['kpi_task_chart_mapping']:
    print(f"  {m['kpi']} → {m['recommended_chart']}")


## Step 4: Fine-Tune with QLoRA

**Expected time**: ~15–25 minutes on T4 GPU

**Watch the loss**: It should decrease from ~2.0–3.0 down to ~0.3–0.8 after 3 epochs.

If you get `CUDA out of memory`, reduce `per_device_train_batch_size` to 1 in config.yaml.

In [ ]:
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name}')
    print(f'VRAM: {vram:.1f} GB')
    if vram < 8:
        print('⚠️  Low VRAM! Set per_device_train_batch_size: 1 in config.yaml')
    else:
        print('✅ VRAM sufficient for training')
else:
    print('❌ No GPU! Go to Runtime > Change runtime type > GPU')


In [ ]:
!python train.py --config config.yaml


In [ ]:
import os
model_dir = './outputs/final_model'
if os.path.exists(model_dir):
    print('✅ Model saved! Files:')
    for fname in os.listdir(model_dir):
        size_mb = os.path.getsize(os.path.join(model_dir, fname)) / 1024 / 1024
        print(f'  {fname} ({size_mb:.1f} MB)')
else:
    print('❌ Model directory not found. Check training output above.')


## Step 5: Inference & Evaluation

In [ ]:
!python inference.py --config config.yaml


In [ ]:
!python inference.py --config config.yaml --evaluate


In [ ]:
# Compare: base model vs. fine-tuned model
print('=' * 60)
print('BASE MODEL (no fine-tuning):')
print('=' * 60)
!python inference.py --config config.yaml --base-only
print('\n' + '=' * 60)
print('FINE-TUNED MODEL:')
print('=' * 60)
!python inference.py --config config.yaml


## Step 6: Save Model to Google Drive (Optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
save_path = '/content/drive/MyDrive/master-thesis/fine-tuned-model'
shutil.copytree('./outputs/final_model', save_path, dirs_exist_ok=True)
print(f'✅ Model saved to Google Drive: {save_path}')


## Troubleshooting & Tips

| Error | Cause | Fix |
|-------|-------|-----|
| `CUDA out of memory` | Not enough VRAM | Set `per_device_train_batch_size: 1` in config.yaml |
| `ModuleNotFoundError: bitsandbytes` | Not installed | Run `!pip install bitsandbytes` |
| `FileNotFoundError: train.jsonl` | Dataset missing | Run Step 3 first |
| `Adapter not found` | Training incomplete | Run Step 4 first |
| Loss not decreasing | LR too low | Try `learning_rate: 5.0e-4` |

### Improving Results:
- Increase `num_train_epochs: 5`
- Increase `num_train_samples: 200`
- Increase LoRA rank: `r: 32`
- Use larger model: `Qwen/Qwen2.5-1.5B-Instruct`

### Understanding Loss:
- Starting loss ~2.0–3.0 is normal
- After 3 epochs: ~0.3–0.8 is good
- Loss < 0.2 on small datasets may indicate overfitting